In [1]:
import os
import json
import numpy as np
from math import exp

In [2]:
# ---------
def resample_sequence(arr, target_T):
    """Resample/truncate arr (T_gt, K, 3) -> target_T frames
       using linear index sampling (no interpolation)."""
    T_gt = arr.shape[0]
    if T_gt == target_T:
        return arr
    if T_gt < 1:
        raise ValueError("Empty sequence")
    idxs = np.linspace(0, T_gt - 1, target_T, dtype=int)
    return arr[idxs]

def mean_joint_distance(pred, gt, joint_weights=None):
    """pred,gt: (T, K, 3). Return scalar mean Euclidean distance (frames x joints averaged).
       Optional joint_weights: (K,) to weight joints (hands > pose etc)"""
    diff = pred - gt                           # (T, K, 3)
    per_joint_frame = np.linalg.norm(diff, axis=-1)  # (T, K)
    if joint_weights is not None:
        jw = np.asarray(joint_weights).reshape(1, -1)
        per_joint_frame = per_joint_frame * jw
        return per_joint_frame.sum() / jw.sum() / per_joint_frame.shape[0]
    return per_joint_frame.mean()

# ------- primary metric -----
def compute_gloss_match_score(
    pred,                 # (T, K, 3) predicted sequence (numpy)
    gt_paths,             # list[str] paths to .npy ground-truth sequences for the gloss
    target_T=None,        # if None, pred.shape[0] used
    mode="best",          # "best" -> min distance across instances, "mean" -> avg across instances
    convert="exp",        # convert distance -> accuracy: "exp" | "threshold" | "linear" | "none"
    alpha=5.0,            # for exp: acc = exp(-dist/alpha)
    threshold=0.5,        # for threshold: dist <= threshold -> acc=1 else 0
    joint_weights=None,   # optional (K,) weights array
    preload_cache=None    # dict path->ndarray to avoid reloading disk every call (optional)
):
    """
    Returns:
      score, best_dist, per_instance_dists
      - score: accuracy-like in [0,1] (unless convert="none")
      - best_dist: the chosen distance (min or mean)
      - per_instance_dists: list of distances for each gt instance
    """
    if target_T is None:
        target_T = pred.shape[0]

    per_instance = []

    for p in gt_paths:
        if preload_cache is not None and p in preload_cache:
            gt_arr = preload_cache[p]
        else:
            gt_arr = np.load(p)   # (T_gt, K, 3)

        # resample/truncate to target_T
        try:
            gt_seq = resample_sequence(gt_arr, target_T)
        except Exception as e:
            continue

        # compute distance
        d = mean_joint_distance(pred, gt_seq, joint_weights)
        per_instance.append(float(d))

    if len(per_instance) == 0:
        return None, None, []

    if mode == "best":
        chosen = float(np.min(per_instance))
    else:  # "mean"
        chosen = float(np.mean(per_instance))

    # convert to accuracy-like
    if convert == "exp":
        score = float(np.exp(-chosen / float(alpha)))
    elif convert == "threshold":
        score = 1.0 if chosen <= threshold else 0.0
    elif convert == "linear":
        # map dist 0->max_dist to [1->0]
        max_dist = max(threshold, 1.0)
        score = max(0.0, 1.0 - chosen / max_dist)
    elif convert == "none":
        score = chosen
    else:
        raise ValueError("unknown convert")

    return score, chosen, per_instance

# ---- build gloss->paths map from annotations -----
def build_gloss_to_paths(annotations_path, data_root):
    # annotations expected to be list of {"id": "1", "gloss": "about", "instances":[{"video_id":"0001"}, ...]}
    with open(annotations_path, "r", encoding="utf-8") as f:
        ann = json.load(f)

    gloss_map = {}
    for entry in ann:
        class_id = str(entry.get("id"))
        gloss = entry.get("gloss")
        if class_id is None or gloss is None:
            continue
        paths = []
        for inst in entry.get("instances", []):
            vid = inst.get("video_id")
            if not vid:
                continue
            fname = vid if vid.endswith(".npy") else f"{vid}.npy"
            p = os.path.join(data_root, class_id, fname)
            if os.path.exists(p):
                paths.append(p)
        if len(paths) > 0:
            gloss_map[gloss] = paths
    return gloss_map


In [5]:
CONFIG = {
    'annotations': r"E:\Balanced_20_Frames_Augmented\NPY\train_final.json",
    'data_root': r'E:\Balanced_20_Frames_Augmented\NPY',

    "custom_accuracy": {
        # enable / disable metric
        "enabled": True,

        # how predicted sequence is compared to GT instances of same gloss
        # "best"  -> min distance across all .npy files of that gloss
        # "mean"  -> mean distance across all .npy files
        "match_mode": "best",

        # how distance is converted to accuracy
        # "exp"        -> exp(-dist / alpha)
        # "threshold"  -> dist <= threshold ? 1 : 0
        # "linear"     -> max(0, 1 - dist / max_dist)
        # "none"       -> raw distance (no accuracy)
        "convert": "exp",

        # exp conversion parameter (higher = more forgiving)
        # typical range: 0.02 – 0.2 if coords are normalized
        "alpha": 0.05,

        # threshold for "threshold" or "linear" mode
        "threshold": 0.1,

        # resampling target
        # usually same as model output length
        "target_frames": 20,

        # whether to weight joints differently
        "use_joint_weights": True,

        # joint weights (length = 75)
        # pose joints lighter, hands heavier
        "joint_weights": {
            "pose": 0.5,     # 33 joints
            "left_hand": 1.0, # 21 joints
            "right_hand": 1.0 # 21 joints
        },

        # cache GT sequences in memory for speed
        "preload_gt": True,

        # compute this metric on validation only
        "compute_on": "val",  # "train" | "val" | "both"

        # log per-gloss breakdown (expensive but informative)
        "log_per_gloss": False
    }
}

cfg = CONFIG

In [ ]:
gloss_map = build_gloss_to_paths(cfg['annotations'], cfg['data_root'])

preload = {}
for gloss, paths in gloss_map.items():
    for p in paths:
        preload[p] = np.load(p)  

# inside validation, per sample:
pred = preds[i].cpu().numpy().reshape(cfg['max_landmark_len'], 75, 3)  # (T,K,3)
gloss = sample['text'] 

paths = gloss_map.get(gloss, [])
score, chosen_dist, per_instance_dists = compute_gloss_match_score(
    pred,
    paths,
    target_T=cfg['max_landmark_len'],
    mode="best",         # best-match across instances
    convert="exp",       # soft accuracy
    alpha=5.0,
    joint_weights=None,
    preload_cache=preload  # or None
)

val_custom_acc_sum += score if score is not None else 0
val_custom_count += 1


NameError: name 'preds' is not defined